# Official workload scaling
The canonical input is `official_token_summary.csv`; the historical alias `motivation_scaling.csv` is not a separate workload.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)
frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists(): frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    order = {'short': 1, 'medium': 2, 'long': 3}
    df['scale_id'] = df['scale'].map(order)
    view = df.groupby(['suite', 'mode', 'scale', 'scale_id'], as_index=False)['total_tokens_mean'].mean().sort_values('scale_id')
    fig, ax = plt.subplots(figsize=(6.8, 3.0), dpi=300)
    for (suite, mode), group in view.groupby(['suite', 'mode']):
        ax.plot(group['scale_id'], group['total_tokens_mean'], marker='o', linewidth=1.0, label=f'{suite}:{mode}')
    ax.set_xticks([1, 2, 3], ['short', 'medium', 'long'])
    ax.set_xlabel('Trajectory length (# calls)')
    ax.set_ylabel('Total tokens (mean)')
    ax.legend(fontsize=6, ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Motivation-Scaling.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Motivation-Scaling.pdf', bbox_inches='tight')
else:
    print('No official summary found; run the official benchmark first.')
# Compatibility filename: motivation_scaling.csv; official_token_summary.csv is authoritative.
